# Post-hoc Reliability Analyses

This notebook operates on the saved prediction artifact from the primary benchmark.
It does **not** retrain the four RPI predictors and does not require raw sequences,
embeddings, checkpoints, or a GPU.

It evaluates protein-balanced performance, failure-risk feature ablations,
secondary-split leakage, leave-one-dataset-out failure transfer, and selective
prediction. The threshold-aligned recomputation in Notebook 03 should be used for
the final confidence/certainty-based reliability results.

In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from __future__ import annotations

from pathlib import Path
import json
import math
import os
import platform
import sys
import time
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
)

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Kaggle input ------------------------------------------------
# The path supplied for the private Kaggle Dataset.
INPUT_ROOT = Path(
    "/kaggle/input/datasets/abdullahnayemwasi/dependable-rpi-journal-artifacts"
)

# Kaggle sometimes exposes a shorter mounted path. The notebook will
# automatically fall back to searching /kaggle/input if needed.
INPUT_FALLBACK_ROOT = Path("/kaggle/input")

# ---- Output ------------------------------------------------------
OUTPUT_ROOT = Path("/kaggle/working/rpi_reliability_posthoc")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FIGURE_ROOT = OUTPUT_ROOT / "figures"
FIGURE_SOURCE_ROOT = FIGURE_ROOT / "source_data"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Reproducibility --------------------------------------------
RANDOM_STATE = 17
N_META_SPLITS = 5

# Random Forest configuration used for the reliability analysis
# protein-grouped failure-detector analysis.
RF_PARAMS = dict(
    n_estimators=350,
    max_depth=6,
    min_samples_leaf=12,
    class_weight="balanced",
    n_jobs=-1,
)

# Optional debugging switch. Keep False for full results.
FAST_DEBUG = False
if FAST_DEBUG:
    RF_PARAMS["n_estimators"] = 40
    print("WARNING: FAST_DEBUG=True. Debug mode is enabled; results are incomplete.")

RUN = {
    "protein_balanced": True,
    "failure_ablation": True,
    "secondary_leakage": True,
    "lodo": True,
    "risk_abstention": True,
    "make_figures": True,
}

COVERAGES = (1.00, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50)


plt.rcParams.update({
    "font.size": 8.5,
    "axes.titlesize": 9,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "figure.dpi": 120,
})

print("Output root:", OUTPUT_ROOT)
print("Python:", sys.version.split()[0])

## 2. Locate the artifact ZIP


In [ ]:
# ============================================================
# 2. Locate artifact ZIP / direct artifact directory
# ============================================================

def locate_artifact_source():
    candidates = []

    if INPUT_ROOT.exists():
        candidates.extend(INPUT_ROOT.rglob("dependable_rpi_journal_v1_ARTIFACTS.zip"))
        candidates.extend(INPUT_ROOT.rglob("*.zip"))

    # Kaggle mount naming sometimes differs from the URL-like path.
    if not candidates and INPUT_FALLBACK_ROOT.exists():
        candidates.extend(
            INPUT_FALLBACK_ROOT.rglob("dependable_rpi_journal_v1_ARTIFACTS.zip")
        )

    if candidates:
        # Prefer the exact expected filename.
        candidates = sorted(
            set(candidates),
            key=lambda p: (p.name != "dependable_rpi_journal_v1_ARTIFACTS.zip", len(str(p)))
        )
        return "zip", candidates[0]

    # Fallback: the dataset may contain extracted files.
    direct_candidates = []
    for root in [INPUT_ROOT, INPUT_FALLBACK_ROOT]:
        if root.exists():
            direct_candidates.extend(root.rglob("all_predictions_enriched.csv"))

    if direct_candidates:
        p = sorted(direct_candidates, key=lambda q: len(str(q)))[0]
        # p = .../analysis/all_predictions_enriched.csv.gz
        return "directory", p.parent.parent

    raise FileNotFoundError(
        "Could not find dependable_rpi_journal_v1_ARTIFACTS.zip or "
        "analysis/all_predictions_enriched.csv.gz under the Kaggle input mounts."
    )

SOURCE_KIND, ARTIFACT_SOURCE = locate_artifact_source()
print("Artifact source type:", SOURCE_KIND)
print("Artifact source:", ARTIFACT_SOURCE)

if SOURCE_KIND == "zip":
    with zipfile.ZipFile(ARTIFACT_SOURCE) as z:
        names = set(z.namelist())
        required = "analysis/all_predictions_enriched.csv"
        if required not in names:
            raise FileNotFoundError(f"Missing required ZIP member: {required}")
        print("ZIP members:", len(names))

## 3. Load only the columns needed for post-hoc analysis

The original enriched prediction table has 43 columns. This notebook intentionally loads only the identifiers, labels/predictions, and **deployable** reliability signals needed here.

Notably excluded from the failure-detector inputs are:

- full benchmark protein degree;
- full benchmark RNA degree;
- any test-label-derived structural quantity.

Those may be useful retrospective diagnostics, but they are not available for a genuinely new prediction and therefore are not valid deployable warning features.

In [ ]:
# ============================================================
# 3. Load compact prediction table
# ============================================================

USECOLS = [
    "row_id", "r_idx", "p_idx",
    "y", "dataset", "protocol", "seed", "fold", "model",
    "run_id", "pair_id",
    "raw_probability", "raw_prediction", "wrong", "raw_confidence",
    "log1p_train_rna_degree", "rna_orphan",
    "rna_centroid_cosine", "protein_centroid_cosine",
    "rna_diagonal_ood", "protein_diagonal_ood",
]

def read_artifact_csv(member: str, **kwargs) -> pd.DataFrame:
    if SOURCE_KIND == "zip":
        with zipfile.ZipFile(ARTIFACT_SOURCE) as z:
            with z.open(member) as f:
                compression = "gzip" if member.endswith(".gz") else None
                return pd.read_csv(f, compression=compression, **kwargs)

    path = ARTIFACT_SOURCE / member
    compression = "gzip" if str(path).endswith(".gz") else None
    return pd.read_csv(path, compression=compression, **kwargs)

pred = read_artifact_csv(
    "analysis/all_predictions_enriched.csv",
    usecols=USECOLS,
)

# Memory-conscious dtypes.
for c in ["seed", "fold", "y", "raw_prediction", "wrong", "rna_orphan"]:
    pred[c] = pd.to_numeric(pred[c], downcast="integer")

for c in [
    "raw_probability", "raw_confidence", "log1p_train_rna_degree",
    "rna_centroid_cosine", "protein_centroid_cosine",
    "rna_diagonal_ood", "protein_diagonal_ood",
]:
    pred[c] = pd.to_numeric(pred[c], downcast="float")

print("Prediction rows:", f"{len(pred):,}")
print("Datasets:", sorted(pred["dataset"].unique()))
print("Models:", sorted(pred["model"].unique()))
print("Protocols:", sorted(pred["protocol"].unique()))
print("Seeds:", sorted(pred["seed"].unique()))
print("Primary run IDs:", pred["run_id"].nunique())

display(
    pred.groupby(["dataset", "model"]).agg(
        rows=("y", "size"),
        runs=("run_id", "nunique"),
        proteins=("p_idx", "nunique"),
        failure_rate=("wrong", "mean"),
    ).round(4)
)

In [ ]:
# ============================================================
# 4. Hard sanity checks before any new analysis
# ============================================================

EXPECTED_DATASETS = {"NPInter2", "NPInter5", "RPI7317"}
EXPECTED_MODELS = {
    "PairMLP-Cross",
    "GraphSAGE-2L",
    "ProtoContrast",
    "ZHMolGraph-Best120",
}

assert set(pred["dataset"].unique()) == EXPECTED_DATASETS
assert set(pred["model"].unique()) == EXPECTED_MODELS
assert set(pred["protocol"].unique()) == {"prot_cold"}
assert pred["run_id"].nunique() == 180, (
    f"Expected 180 primary runs; found {pred['run_id'].nunique()}."
)
assert not pred[["run_id", "row_id"]].duplicated().any(), (
    "run_id + row_id must uniquely identify a saved prediction."
)
assert set(pred["y"].unique()).issubset({0, 1})
assert set(pred["wrong"].unique()).issubset({0, 1})

required_feature_cols = [
    "raw_confidence",
    "log1p_train_rna_degree",
    "rna_centroid_cosine",
    "protein_centroid_cosine",
    "rna_diagonal_ood",
    "protein_diagonal_ood",
]
if pred[required_feature_cols].replace([np.inf, -np.inf], np.nan).isna().any().any():
    raise ValueError("Deployable feature columns contain NaN or infinite values.")

if len(pred) != 449_244:
    print(
        f"NOTE: expected 449,244 rows from the frozen journal artifact, "
        f"but found {len(pred):,}. Continue only if this is intentional."
    )
else:
    print("Row-count audit passed: 449,244 predictions.")

print("Sanity checks passed.")

# Shared helper functions

Several methodological details are enforced here:

- all failure-detector comparisons use the **same RF hyperparameters**;
- feature ablations within a dataset/model use the **same protein-grouped meta-folds**;
- `wrong=1` is the positive class for failure detection;
- all primary RPI models remain untouched;
- every manuscript figure saves the exact source CSV used to draw it.

In [ ]:
# ============================================================
# 5. Shared helpers
# ============================================================

EPS = 1e-12

def safe_auroc(y, score, sample_weight=None):
    y = np.asarray(y)
    if np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, score, sample_weight=sample_weight))

def safe_auprc(y, score, sample_weight=None):
    y = np.asarray(y)
    if np.unique(y).size < 2:
        return np.nan
    return float(average_precision_score(y, score, sample_weight=sample_weight))

def weighted_mcc(y, pred_y, weights):
    y = np.asarray(y, dtype=int)
    pred_y = np.asarray(pred_y, dtype=int)
    w = np.asarray(weights, dtype=float)

    tp = w[(y == 1) & (pred_y == 1)].sum()
    tn = w[(y == 0) & (pred_y == 0)].sum()
    fp = w[(y == 0) & (pred_y == 1)].sum()
    fn = w[(y == 1) & (pred_y == 0)].sum()

    den = math.sqrt(
        max((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn), 0.0)
    )
    if den <= EPS:
        return np.nan
    return float((tp * tn - fp * fn) / den)

def edge_metrics(g):
    y = g["y"].to_numpy(int)
    p = g["raw_probability"].to_numpy(float)
    yh = g["raw_prediction"].to_numpy(int)
    return {
        "AUROC": safe_auroc(y, p),
        "AUPRC": safe_auprc(y, p),
        "MCC": float(matthews_corrcoef(y, yh)),
        "ERROR": float(np.mean(y != yh)),
        "BRIER": float(np.mean((p - y) ** 2)),
    }

def protein_balanced_metrics(g):
    # Each held-out protein contributes total weight 1, regardless of how many
    # edges it contributes to the fold.
    counts = g.groupby("p_idx")["p_idx"].transform("size").to_numpy(float)
    w = 1.0 / counts

    y = g["y"].to_numpy(int)
    p = g["raw_probability"].to_numpy(float)
    yh = g["raw_prediction"].to_numpy(int)

    return {
        "AUROC": safe_auroc(y, p, sample_weight=w),
        "AUPRC": safe_auprc(y, p, sample_weight=w),
        "MCC": weighted_mcc(y, yh, w),
        "ERROR": float(np.average(y != yh, weights=w)),
        "BRIER": float(np.average((p - y) ** 2, weights=w)),
    }

def macro_per_protein_metrics(g):
    rows = []
    for protein, pg in g.groupby("p_idx"):
        y = pg["y"].to_numpy(int)
        p = pg["raw_probability"].to_numpy(float)
        yh = pg["raw_prediction"].to_numpy(int)
        both_classes = np.unique(y).size == 2

        rows.append({
            "p_idx": protein,
            "n_edges": len(pg),
            "positive_prevalence": float(y.mean()),
            "eligible_for_auc": int(both_classes),
            "protein_AUROC": safe_auroc(y, p) if both_classes else np.nan,
            "protein_AUPRC": safe_auprc(y, p) if both_classes else np.nan,
            "protein_ERROR": float(np.mean(y != yh)),
            "protein_BRIER": float(np.mean((p - y) ** 2)),
        })

    per_protein = pd.DataFrame(rows)
    return {
        "n_proteins": int(len(per_protein)),
        "n_auc_eligible_proteins": int(per_protein["eligible_for_auc"].sum()),
        "macro_protein_AUROC": float(per_protein["protein_AUROC"].mean()),
        "macro_protein_AUPRC": float(per_protein["protein_AUPRC"].mean()),
        "macro_protein_ERROR": float(per_protein["protein_ERROR"].mean()),
        "macro_protein_BRIER": float(per_protein["protein_BRIER"].mean()),
    }

def holm_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    out = np.full_like(p, np.nan)
    valid = np.where(np.isfinite(p))[0]
    if len(valid) == 0:
        return out

    order = valid[np.argsort(p[valid])]
    m = len(order)
    prev = 0.0
    for rank, idx in enumerate(order):
        adj = min(1.0, (m - rank) * p[idx])
        adj = max(prev, adj)
        out[idx] = adj
        prev = adj
    return out

def new_rf(seed):
    return RandomForestClassifier(random_state=seed, **RF_PARAMS)

def failure_metrics(y, risk):
    y = np.asarray(y, dtype=int)
    risk = np.asarray(risk, dtype=float)
    order = np.argsort(-risk)
    k10 = max(1, int(math.ceil(0.10 * len(y))))
    k20 = max(1, int(math.ceil(0.20 * len(y))))
    base = float(y.mean())
    top10 = float(y[order[:k10]].mean())
    top20 = float(y[order[:k20]].mean())

    return {
        "n": int(len(y)),
        "baseline_failure_rate": base,
        "failure_AUROC": safe_auroc(y, risk),
        "failure_AUPRC": safe_auprc(y, risk),
        "top10_failure_rate": top10,
        "top20_failure_rate": top20,
        "top10_enrichment": float(top10 / max(base, EPS)),
    }

def make_failure_splits(g, strategy, n_splits=N_META_SPLITS):
    y = g["wrong"].to_numpy(int)

    if strategy == "record":
        splitter = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
        return list(splitter.split(np.zeros(len(g)), y))

    if strategy == "pair":
        groups = g["pair_id"].astype(str).to_numpy()
    elif strategy == "protein":
        groups = (
            g["dataset"].astype(str) + "|p" + g["p_idx"].astype(str)
        ).to_numpy()
    else:
        raise ValueError(strategy)

    n_groups = len(np.unique(groups))
    n_splits = min(n_splits, n_groups)
    if n_splits < 2:
        raise ValueError(f"Not enough groups for {strategy} splitting.")

    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    return list(splitter.split(np.zeros(len(g)), y, groups))

def save_figure_with_source(
    fig,
    source_df,
    figure_id,
    caption,
    interpretation,
):
    pdf = FIGURE_ROOT / f"{figure_id}.pdf"
    png = FIGURE_ROOT / f"{figure_id}.png"
    src = FIGURE_SOURCE_ROOT / f"{figure_id}_source.csv"

    source_df.to_csv(src, index=False)
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=400, bbox_inches="tight")
    plt.close(fig)

    manifest_path = FIGURE_ROOT / "figure_manifest.csv"
    row = pd.DataFrame([{
        "figure_id": figure_id,
        "source_csv": src.name,
        "pdf": pdf.name,
        "png": png.name,
        "caption": caption,
        "interpretation": interpretation,
    }])

    if manifest_path.exists():
        old = pd.read_csv(manifest_path)
        old = old[old["figure_id"] != figure_id]
        row = pd.concat([old, row], ignore_index=True)

    row.to_csv(manifest_path, index=False)

def run_oof_failure_detector(g, features, splits):
    y = g["wrong"].to_numpy(int)
    X = g[features].to_numpy(float)

    oof = np.full(len(g), np.nan, dtype=float)

    for cv_fold, (tr, te) in enumerate(splits):
        if np.unique(y[tr]).size < 2:
            oof[te] = y[tr].mean()
            continue

        clf = new_rf(RANDOM_STATE + cv_fold)
        clf.fit(X[tr], y[tr])
        oof[te] = clf.predict_proba(X[te])[:, 1]

    if not np.isfinite(oof).all():
        raise RuntimeError("OOF failure scores contain missing values.")

    return oof

# Experiment 1 — Protein-balanced versus edge-weighted evaluation

## Why?

Ordinary test metrics give every interaction edge equal weight. A held-out protein with 1,000 test edges therefore influences the aggregate score roughly 1,000 times more than a protein with one edge.

That is not necessarily wrong, but it answers an **edge-centric** question.

We also evaluate the corresponding **entity-centric** question:

> What would performance look like if each held-out protein contributed equal total weight?

For a fold containing protein \(p\) with \(n_p\) test edges, every edge belonging to that protein receives:

\[
w_i = \frac{1}{n_p}.
\]

Therefore each protein contributes total weight 1.

We compute:

- conventional edge-weighted AUROC, AUPRC, MCC, error, and Brier;
- equal-protein-weighted versions of the same metrics;
- macro per-protein AUROC/AUPRC for proteins containing both positive and negative labels;
- number of proteins for which per-protein AUROC is mathematically defined.

The primary model predictions are **not changed**.

In [ ]:
# ============================================================
# 6. Experiment 1: protein-balanced evaluation
# ============================================================

PB_ROOT = OUTPUT_ROOT / "01_protein_balanced"
PB_ROOT.mkdir(parents=True, exist_ok=True)

if RUN["protein_balanced"]:
    run_rows = []
    per_protein_rows = []

    group_cols = ["dataset", "model", "seed", "fold", "run_id"]

    for keys, g in pred.groupby(group_cols, sort=True):
        dataset, model, seed, fold, run_id = keys

        edge = edge_metrics(g)
        balanced = protein_balanced_metrics(g)
        macro = macro_per_protein_metrics(g)

        row = {
            "dataset": dataset,
            "model": model,
            "seed": int(seed),
            "fold": int(fold),
            "run_id": run_id,
            "n_edges": int(len(g)),
            "n_proteins": int(g["p_idx"].nunique()),
            **{f"edge_{k}": v for k, v in edge.items()},
            **{f"protein_balanced_{k}": v for k, v in balanced.items()},
            **macro,
        }

        for metric in ["AUROC", "AUPRC", "MCC", "ERROR", "BRIER"]:
            row[f"delta_{metric}_PB_minus_edge"] = (
                row[f"protein_balanced_{metric}"] - row[f"edge_{metric}"]
            )

        run_rows.append(row)

        # Save per-protein metrics as an audit table.
        for protein, pg in g.groupby("p_idx"):
            y = pg["y"].to_numpy(int)
            p = pg["raw_probability"].to_numpy(float)
            yh = pg["raw_prediction"].to_numpy(int)
            eligible = np.unique(y).size == 2

            per_protein_rows.append({
                "dataset": dataset,
                "model": model,
                "seed": int(seed),
                "fold": int(fold),
                "run_id": run_id,
                "p_idx": protein,
                "n_edges": int(len(pg)),
                "positive_prevalence": float(y.mean()),
                "eligible_for_auc": int(eligible),
                "protein_AUROC": safe_auroc(y, p) if eligible else np.nan,
                "protein_AUPRC": safe_auprc(y, p) if eligible else np.nan,
                "protein_ERROR": float(np.mean(y != yh)),
                "protein_BRIER": float(np.mean((p - y) ** 2)),
            })

    pb_runs = pd.DataFrame(run_rows)
    pb_proteins = pd.DataFrame(per_protein_rows)

    pb_runs.to_csv(
        PB_ROOT / "TABLE_run_level_edge_vs_protein_balanced.csv",
        index=False,
    )
    pb_proteins.to_csv(
        PB_ROOT / "TABLE_per_protein_metrics.csv.gz",
        index=False,
        compression="gzip",
    )

    metric_cols = [
        "edge_AUROC", "protein_balanced_AUROC", "macro_protein_AUROC",
        "edge_AUPRC", "protein_balanced_AUPRC", "macro_protein_AUPRC",
        "edge_MCC", "protein_balanced_MCC",
        "edge_ERROR", "protein_balanced_ERROR", "macro_protein_ERROR",
        "edge_BRIER", "protein_balanced_BRIER", "macro_protein_BRIER",
        "n_proteins", "n_auc_eligible_proteins",
    ]

    pb_summary = (
        pb_runs
        .groupby(["dataset", "model"])[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )
    pb_summary.columns = [
        "__".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple) else col
        for col in pb_summary.columns
    ]
    pb_summary.to_csv(
        PB_ROOT / "TABLE_summary_edge_vs_protein_balanced.csv",
        index=False,
    )

    # Paired run-level tests: protein-balanced vs edge-weighted.
    test_rows = []
    for (dataset, model), g in pb_runs.groupby(["dataset", "model"]):
        for metric in ["AUROC", "AUPRC", "MCC", "ERROR", "BRIER"]:
            a = g[f"edge_{metric}"].to_numpy(float)
            b = g[f"protein_balanced_{metric}"].to_numpy(float)
            valid = np.isfinite(a) & np.isfinite(b)

            if valid.sum() >= 3 and np.any(np.abs(b[valid] - a[valid]) > 0):
                stat, pval = wilcoxon(b[valid], a[valid], zero_method="wilcox")
            else:
                stat, pval = np.nan, np.nan

            test_rows.append({
                "dataset": dataset,
                "model": model,
                "metric": metric,
                "n_runs": int(valid.sum()),
                "edge_mean": float(np.nanmean(a)),
                "protein_balanced_mean": float(np.nanmean(b)),
                "mean_delta_PB_minus_edge": float(np.nanmean(b - a)),
                "wilcoxon_stat": stat,
                "p_value": pval,
            })

    pb_tests = pd.DataFrame(test_rows)
    pb_tests["p_holm"] = np.nan
    for _, idx in pb_tests.groupby(["dataset", "metric"]).groups.items():
        pb_tests.loc[idx, "p_holm"] = holm_adjust(
            pb_tests.loc[idx, "p_value"].to_numpy(float)
        )

    pb_tests.to_csv(
        PB_ROOT / "TABLE_paired_tests_edge_vs_protein_balanced.csv",
        index=False,
    )

    print("Protein-balanced experiment complete.")
    display(
        pb_tests[pb_tests["metric"].isin(["AUROC", "MCC", "ERROR"])]
        .sort_values(["dataset", "metric", "model"])
        .round(4)
    )

In [ ]:
# ============================================================
# 7. Figure: change in AUROC after equal-protein weighting
# ============================================================

if RUN["protein_balanced"] and RUN["make_figures"]:
    src = (
        pb_runs.groupby(["dataset", "model"])
        .agg(
            edge_AUROC=("edge_AUROC", "mean"),
            protein_balanced_AUROC=("protein_balanced_AUROC", "mean"),
        )
        .reset_index()
    )
    src["delta_AUROC"] = (
        src["protein_balanced_AUROC"] - src["edge_AUROC"]
    )
    src["label"] = src["dataset"] + " / " + src["model"].str.replace(
        "ZHMolGraph-Best120", "ZHMolGraph", regex=False
    )

    src = src.sort_values("delta_AUROC")
    fig, ax = plt.subplots(figsize=(3.45, 4.1))
    y = np.arange(len(src))
    ax.barh(y, src["delta_AUROC"])
    ax.axvline(0, linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(src["label"])
    ax.set_xlabel("Protein-balanced AUROC − edge-weighted AUROC")
    ax.set_title("Effect of equal protein weighting")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    save_figure_with_source(
        fig,
        src.drop(columns="label"),
        "FIG_PB_AUROC_DELTA",
        caption=(
            "Change in protein-cold AUROC when each held-out protein is "
            "given equal total weight instead of weighting every interaction "
            "edge equally. Positive values indicate that high-edge-count "
            "proteins depress the conventional edge-weighted AUROC; negative "
            "values indicate the opposite."
        ),
        interpretation=(
            "This figure tests whether reported protein-cold performance is "
            "sensitive to the statistical unit of evaluation: interactions "
            "versus held-out proteins."
        ),
    )

# Experiment 2 — Strict failure-detector ablation

## The question

Does information **outside the primary model's own confidence** help identify which RPI predictions are likely to be wrong?

The failure target is:

\[
F_i = \mathbf{1}[\hat{Y}_i \neq Y_i].
\]

The secondary Random Forest outputs:

\[
q_i = P(F_i=1 \mid \text{deployable signals}).
\]

## Feature sets

To avoid pretending that confidence, margin, and entropy are independent information sources, the main ablation uses only one probability-derived certainty feature: `raw_confidence`.

The four feature sets are:

1. **Confidence**
   - `raw_confidence`

2. **Confidence + support**
   - `raw_confidence`
   - `log1p_train_rna_degree`

3. **Confidence + OOD/familiarity**
   - `raw_confidence`
   - RNA/protein centroid cosine
   - RNA/protein diagonal OOD distance

4. **Confidence + support + OOD/familiarity**
   - all of the above

`rna_orphan` is not needed as a separate central feature because it is exactly the zero-support state of RNA training degree. Full protein degree is excluded because it is retrospective/oracle information.

## Validation

Every feature set is evaluated on the **same protein-grouped secondary folds**. Thus no held-out protein can appear in both secondary training and evaluation.

In [ ]:
# ============================================================
# 8. Experiment 2: strict deployable feature ablation
# ============================================================

ABLATION_ROOT = OUTPUT_ROOT / "02_failure_ablation"
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)

ABLATION_FEATURES = {
    "confidence": [
        "raw_confidence",
    ],
    "confidence+support": [
        "raw_confidence",
        "log1p_train_rna_degree",
    ],
    "confidence+ood": [
        "raw_confidence",
        "rna_centroid_cosine",
        "protein_centroid_cosine",
        "rna_diagonal_ood",
        "protein_diagonal_ood",
    ],
    "confidence+support+ood": [
        "raw_confidence",
        "log1p_train_rna_degree",
        "rna_centroid_cosine",
        "protein_centroid_cosine",
        "rna_diagonal_ood",
        "protein_diagonal_ood",
    ],
}

FINAL_FAILURE_FEATURE_SET = "confidence+support+ood"
FINAL_FAILURE_FEATURES = ABLATION_FEATURES[FINAL_FAILURE_FEATURE_SET]

if RUN["failure_ablation"]:
    ablation_rows = []
    ablation_scores = []

    for (dataset, model), g0 in pred.groupby(["dataset", "model"], sort=True):
        g = g0.reset_index(drop=True).copy()
        y = g["wrong"].to_numpy(int)

        # Generate ONCE so every ablation uses identical protein-grouped folds.
        splits = make_failure_splits(g, "protein")

        for feature_name, features in ABLATION_FEATURES.items():
            risk = run_oof_failure_detector(g, features, splits)
            metrics = failure_metrics(y, risk)

            ablation_rows.append({
                "dataset": dataset,
                "model": model,
                "feature_set": feature_name,
                "split_strategy": "protein_grouped",
                **metrics,
            })

            z = g[
                [
                    "dataset", "model", "seed", "fold", "run_id",
                    "row_id", "pair_id", "r_idx", "p_idx",
                    "y", "raw_probability", "raw_prediction",
                    "wrong", "raw_confidence",
                ]
            ].copy()
            z["feature_set"] = feature_name
            z["failure_risk"] = risk
            ablation_scores.append(z)

        print("Finished:", dataset, "/", model)

    failure_ablation = pd.DataFrame(ablation_rows)
    failure_ablation_oof = pd.concat(ablation_scores, ignore_index=True)

    # Add improvement relative to confidence only.
    base = (
        failure_ablation[failure_ablation["feature_set"] == "confidence"]
        [["dataset", "model", "failure_AUROC", "failure_AUPRC"]]
        .rename(columns={
            "failure_AUROC": "confidence_failure_AUROC",
            "failure_AUPRC": "confidence_failure_AUPRC",
        })
    )

    failure_ablation = failure_ablation.merge(
        base, on=["dataset", "model"], how="left"
    )
    failure_ablation["delta_AUROC_vs_confidence"] = (
        failure_ablation["failure_AUROC"]
        - failure_ablation["confidence_failure_AUROC"]
    )
    failure_ablation["delta_AUPRC_vs_confidence"] = (
        failure_ablation["failure_AUPRC"]
        - failure_ablation["confidence_failure_AUPRC"]
    )

    failure_ablation.to_csv(
        ABLATION_ROOT / "TABLE_failure_ablation_protein_grouped.csv",
        index=False,
    )
    failure_ablation_oof.to_csv(
        ABLATION_ROOT / "failure_ablation_oof_scores.csv.gz",
        index=False,
        compression="gzip",
    )

    display(
        failure_ablation[
            [
                "dataset", "model", "feature_set",
                "failure_AUROC", "failure_AUPRC",
                "top10_failure_rate", "top10_enrichment",
                "delta_AUROC_vs_confidence",
            ]
        ].round(4)
    )

In [ ]:
# ============================================================
# 9. Figures: strict failure-detector ablation
# ============================================================

if RUN["failure_ablation"] and RUN["make_figures"]:
    feature_order = [
        "confidence",
        "confidence+support",
        "confidence+ood",
        "confidence+support+ood",
    ]

    for dataset, dg in failure_ablation.groupby("dataset"):
        src = dg.copy()
        src["feature_set"] = pd.Categorical(
            src["feature_set"],
            categories=feature_order,
            ordered=True,
        )
        src = src.sort_values(["feature_set", "model"])

        fig, ax = plt.subplots(figsize=(3.45, 2.8))
        x = np.arange(len(feature_order))

        for model, mg in src.groupby("model", observed=True):
            mg = mg.set_index("feature_set").reindex(feature_order)
            ax.plot(
                x,
                mg["failure_AUROC"],
                marker="o",
                linewidth=1.4,
                label=model.replace("ZHMolGraph-Best120", "ZHMolGraph"),
            )

        ax.axhline(0.5, linestyle="--", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(
            ["Conf.", "+Support", "+OOD", "+Both"],
            rotation=0,
        )
        ax.set_ylabel("Failure-detection AUROC")
        ax.set_title(f"{dataset}: deployable failure-signal ablation")
        ax.set_ylim(0.35, 0.95)
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)
        fig.tight_layout()

        save_figure_with_source(
            fig,
            src,
            f"FIG_FAILURE_ABLATION_{dataset}",
            caption=(
                f"Protein-grouped failure-detection AUROC on {dataset} as "
                "deployable information is added to primary-model confidence. "
                "All feature sets use identical secondary cross-validation "
                "folds and Random Forest hyperparameters."
            ),
            interpretation=(
                "Improvement beyond the confidence-only condition demonstrates "
                "that training support and/or representation familiarity contain "
                "failure information not already captured by model confidence."
            ),
        )

# Experiment 3 — Secondary-evaluation leakage analysis

## Why?

The primary RPI predictor is already evaluated under protein-cold splitting. However, a **second-level failure detector can still be evaluated too easily** if its own train/test split ignores biological identity.

We therefore compare the same final failure detector under three secondary validation schemes:

1. **Record-level stratified split**  
   Individual saved prediction records are split randomly. Because the same biological pair is predicted across multiple seeds, near-duplicate biological cases can occur on both sides.

2. **Pair-grouped split**  
   All predictions for the same RNA-protein pair stay together. This prevents exact-pair overlap, but a protein may still appear in both detector training and detector evaluation.

3. **Protein-grouped split**  
   All predictions involving the same held-out protein stay together. This matches the deployment question of recognizing risk for a protein the reliability model has not already seen.

The record-level and pair-grouped conditions are **diagnostic controls**, not recommended deployment evaluations.

In [ ]:
# ============================================================
# 10. Experiment 3: secondary split / leakage comparison
# ============================================================

LEAK_ROOT = OUTPUT_ROOT / "03_secondary_leakage"
LEAK_ROOT.mkdir(parents=True, exist_ok=True)

SPLIT_STRATEGIES = ["record", "pair", "protein"]

if RUN["secondary_leakage"]:
    leakage_rows = []
    leakage_scores = []

    for (dataset, model), g0 in pred.groupby(["dataset", "model"], sort=True):
        g = g0.reset_index(drop=True).copy()
        y = g["wrong"].to_numpy(int)

        for strategy in SPLIT_STRATEGIES:
            splits = make_failure_splits(g, strategy)
            risk = run_oof_failure_detector(
                g,
                FINAL_FAILURE_FEATURES,
                splits,
            )
            metrics = failure_metrics(y, risk)

            leakage_rows.append({
                "dataset": dataset,
                "model": model,
                "feature_set": FINAL_FAILURE_FEATURE_SET,
                "split_strategy": strategy,
                **metrics,
            })

            z = g[
                [
                    "dataset", "model", "seed", "fold", "run_id",
                    "row_id", "pair_id", "p_idx", "y", "wrong",
                    "raw_probability", "raw_prediction", "raw_confidence",
                ]
            ].copy()
            z["split_strategy"] = strategy
            z["failure_risk"] = risk
            leakage_scores.append(z)

        print("Finished:", dataset, "/", model)

    leakage_results = pd.DataFrame(leakage_rows)
    leakage_oof = pd.concat(leakage_scores, ignore_index=True)

    # Quantify inflation relative to the deployment-aligned protein grouping.
    protein_ref = (
        leakage_results[leakage_results["split_strategy"] == "protein"]
        [["dataset", "model", "failure_AUROC", "failure_AUPRC"]]
        .rename(columns={
            "failure_AUROC": "protein_grouped_AUROC",
            "failure_AUPRC": "protein_grouped_AUPRC",
        })
    )
    leakage_results = leakage_results.merge(
        protein_ref, on=["dataset", "model"], how="left"
    )
    leakage_results["AUROC_minus_protein_grouped"] = (
        leakage_results["failure_AUROC"]
        - leakage_results["protein_grouped_AUROC"]
    )
    leakage_results["AUPRC_minus_protein_grouped"] = (
        leakage_results["failure_AUPRC"]
        - leakage_results["protein_grouped_AUPRC"]
    )

    leakage_results.to_csv(
        LEAK_ROOT / "TABLE_secondary_split_comparison.csv",
        index=False,
    )
    leakage_oof.to_csv(
        LEAK_ROOT / "secondary_split_oof_scores.csv.gz",
        index=False,
        compression="gzip",
    )

    display(
        leakage_results[
            [
                "dataset", "model", "split_strategy",
                "failure_AUROC", "failure_AUPRC",
                "top10_enrichment", "AUROC_minus_protein_grouped",
            ]
        ].round(4)
    )

In [ ]:
# ============================================================
# 11. Figures: secondary split comparison
# ============================================================

if RUN["secondary_leakage"] and RUN["make_figures"]:
    strategy_order = ["record", "pair", "protein"]

    for dataset, dg in leakage_results.groupby("dataset"):
        src = dg.copy()
        src["split_strategy"] = pd.Categorical(
            src["split_strategy"],
            categories=strategy_order,
            ordered=True,
        )
        src = src.sort_values(["split_strategy", "model"])

        fig, ax = plt.subplots(figsize=(3.45, 2.8))
        x = np.arange(len(strategy_order))

        for model, mg in src.groupby("model", observed=True):
            mg = mg.set_index("split_strategy").reindex(strategy_order)
            ax.plot(
                x,
                mg["failure_AUROC"],
                marker="o",
                linewidth=1.4,
                label=model.replace("ZHMolGraph-Best120", "ZHMolGraph"),
            )

        ax.axhline(0.5, linestyle="--", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(["Record", "Pair", "Protein"])
        ax.set_ylabel("Failure-detection AUROC")
        ax.set_title(f"{dataset}: effect of secondary split unit")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False)
        fig.tight_layout()

        save_figure_with_source(
            fig,
            src,
            f"FIG_SECONDARY_LEAKAGE_{dataset}",
            caption=(
                f"Failure-detection AUROC on {dataset} under progressively "
                "stricter secondary validation. Record-level splitting allows "
                "repeated biological cases to cross folds; pair grouping blocks "
                "exact-pair overlap; protein grouping prevents any held-out "
                "protein from occurring in both reliability-model training and "
                "evaluation."
            ),
            interpretation=(
                "A drop from record/pair validation to protein-grouped validation "
                "quantifies how much apparent failure detectability depends on "
                "biological identity leakage at the secondary model level."
            ),
        )

# Experiment 4 — Standardized leave-one-dataset-out (LODO) failure transfer

The input artifact contains an earlier LODO analysis produced with a slightly different Random Forest configuration and a broader feature set.

Here, LODO uses the **same deployable feature set and RF configuration** as the strict within-dataset detector.

For each base RPI model:

- train the failure detector on two datasets;
- evaluate directly on the third;
- do not use target-dataset failure labels for fitting.

This tests whether **failure signatures themselves transfer across RPI benchmarks**.

In [ ]:
# ============================================================
# 12. Experiment 4: standardized LODO
# ============================================================

LODO_ROOT = OUTPUT_ROOT / "04_lodo"
LODO_ROOT.mkdir(parents=True, exist_ok=True)

if RUN["lodo"]:
    lodo_rows = []
    lodo_scores = []

    for model, gm in pred.groupby("model", sort=True):
        datasets = sorted(gm["dataset"].unique())

        for target_dataset in datasets:
            tr = gm[gm["dataset"] != target_dataset].copy()
            te = gm[gm["dataset"] == target_dataset].copy()

            Xtr = tr[FINAL_FAILURE_FEATURES].to_numpy(float)
            Xte = te[FINAL_FAILURE_FEATURES].to_numpy(float)
            ytr = tr["wrong"].to_numpy(int)
            yte = te["wrong"].to_numpy(int)

            clf = new_rf(RANDOM_STATE)
            clf.fit(Xtr, ytr)
            risk = clf.predict_proba(Xte)[:, 1]

            metrics = failure_metrics(yte, risk)

            lodo_rows.append({
                "model": model,
                "target_dataset": target_dataset,
                "train_datasets": " + ".join(
                    [d for d in datasets if d != target_dataset]
                ),
                "feature_set": FINAL_FAILURE_FEATURE_SET,
                **metrics,
            })

            z = te[
                [
                    "dataset", "model", "seed", "fold", "run_id",
                    "row_id", "pair_id", "p_idx", "y", "wrong",
                ]
            ].copy()
            z["target_dataset"] = target_dataset
            z["failure_risk_lodo"] = risk
            lodo_scores.append(z)

    lodo_results = pd.DataFrame(lodo_rows)
    lodo_oof = pd.concat(lodo_scores, ignore_index=True)

    lodo_results.to_csv(
        LODO_ROOT / "TABLE_lodo_final_features.csv",
        index=False,
    )
    lodo_oof.to_csv(
        LODO_ROOT / "lodo_failure_scores.csv.gz",
        index=False,
        compression="gzip",
    )

    display(lodo_results.round(4))

In [ ]:
# ============================================================
# 13. Figure: LODO failure transfer
# ============================================================

if RUN["lodo"] and RUN["make_figures"]:
    src = lodo_results.copy()
    dataset_order = ["NPInter2", "NPInter5", "RPI7317"]
    src["target_dataset"] = pd.Categorical(
        src["target_dataset"],
        categories=dataset_order,
        ordered=True,
    )
    src = src.sort_values(["target_dataset", "model"])

    fig, ax = plt.subplots(figsize=(3.45, 2.8))
    x = np.arange(len(dataset_order))

    for model, mg in src.groupby("model", observed=True):
        mg = mg.set_index("target_dataset").reindex(dataset_order)
        ax.plot(
            x,
            mg["failure_AUROC"],
            marker="o",
            linewidth=1.4,
            label=model.replace("ZHMolGraph-Best120", "ZHMolGraph"),
        )

    ax.axhline(0.5, linestyle="--", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(dataset_order)
    ax.set_ylabel("Failure-detection AUROC")
    ax.set_title("Leave-one-dataset-out failure transfer")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()

    save_figure_with_source(
        fig,
        src,
        "FIG_LODO_FAILURE_TRANSFER",
        caption=(
            "Leave-one-dataset-out transfer of the deployable failure detector. "
            "For each target dataset, the reliability model is trained only on "
            "predictions from the other two datasets. The primary RPI model is "
            "not retrained."
        ),
        interpretation=(
            "Values above 0.5 indicate that some failure signatures transfer "
            "across benchmarks. Weak or reversed transfer shows that reliability "
            "models themselves are sensitive to domain shift."
        ),
    )

# Experiment 5 — Does learned failure risk improve selective prediction?

A secondary failure detector is operationally useful only if its risk score can support better decisions.

We therefore compare two abstention policies on the **same saved RPI predictions**:

### Confidence-based abstention
Reject predictions with the lowest primary-model confidence.

\[
r_i^{conf} = 1 - confidence_i
\]

### Learned-risk abstention
Reject predictions with the largest strict protein-grouped Random Forest failure risk.

\[
r_i^{learned} = P(F_i=1 \mid confidence,\ support,\ OOD)
\]

At each coverage, we keep the safest fraction and recompute:

- MCC;
- error;
- AUROC;
- AUPRC.

The comparison is performed **within each original seed/fold run**, preserving the 15 primary experimental units per dataset/model.

In [ ]:
# ============================================================
# 14. Experiment 5: confidence vs learned-risk abstention
# ============================================================

RISK_ROOT = OUTPUT_ROOT / "05_risk_abstention"
RISK_ROOT.mkdir(parents=True, exist_ok=True)

def selective_metrics_for_order(g, risk, coverages=COVERAGES):
    order = np.argsort(np.asarray(risk, dtype=float))  # lower = safer
    rows = []

    for coverage in coverages:
        k = max(1, int(math.ceil(float(coverage) * len(g))))
        idx = order[:k]
        accepted = g.iloc[idx]

        y = accepted["y"].to_numpy(int)
        p = accepted["raw_probability"].to_numpy(float)
        yh = accepted["raw_prediction"].to_numpy(int)

        rows.append({
            "coverage": float(coverage),
            "accepted_n": int(k),
            "AUROC": safe_auroc(y, p),
            "AUPRC": safe_auprc(y, p),
            "MCC": float(matthews_corrcoef(y, yh)),
            "ERROR": float(np.mean(y != yh)),
        })

    return rows

if RUN["risk_abstention"]:
    if "failure_ablation_oof" not in globals():
        score_path = ABLATION_ROOT / "failure_ablation_oof_scores.csv.gz"
        if not score_path.exists():
            raise RuntimeError(
                "Run Experiment 2 first; strict protein-grouped OOF failure "
                "scores are required for learned-risk abstention."
            )
        failure_ablation_oof = pd.read_csv(score_path)

    learned = failure_ablation_oof[
        failure_ablation_oof["feature_set"] == FINAL_FAILURE_FEATURE_SET
    ].copy()

    # Verify exact one-to-one coverage of primary predictions.
    key = ["run_id", "row_id"]
    assert not learned[key].duplicated().any()
    assert len(learned) == len(pred)

    risk_rows = []

    for run_id, g0 in learned.groupby("run_id", sort=True):
        g = g0.reset_index(drop=True).copy()

        common = {
            "dataset": g["dataset"].iloc[0],
            "model": g["model"].iloc[0],
            "seed": int(g["seed"].iloc[0]),
            "fold": int(g["fold"].iloc[0]),
            "run_id": run_id,
        }

        confidence_risk = 1.0 - g["raw_confidence"].to_numpy(float)
        learned_risk = g["failure_risk"].to_numpy(float)

        for row in selective_metrics_for_order(g, confidence_risk):
            risk_rows.append({
                **common,
                "risk_method": "confidence",
                **row,
            })

        for row in selective_metrics_for_order(g, learned_risk):
            risk_rows.append({
                **common,
                "risk_method": "learned_failure_risk",
                **row,
            })

    risk_coverage = pd.DataFrame(risk_rows)
    risk_coverage.to_csv(
        RISK_ROOT / "risk_coverage_confidence_vs_learned.csv",
        index=False,
    )

    risk_summary = (
        risk_coverage
        .groupby(["dataset", "model", "risk_method", "coverage"])
        [["MCC", "ERROR", "AUROC", "AUPRC"]]
        .agg(["mean", "std"])
        .reset_index()
    )
    risk_summary.columns = [
        "__".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple) else col
        for col in risk_summary.columns
    ]
    risk_summary.to_csv(
        RISK_ROOT / "TABLE_risk_coverage_summary.csv",
        index=False,
    )

    # Paired run-level comparison between the two abstention policies.
    compare_rows = []
    for (dataset, model, coverage), dg in risk_coverage.groupby(
        ["dataset", "model", "coverage"]
    ):
        a = (
            dg[dg["risk_method"] == "confidence"]
            [["seed", "fold", "MCC", "ERROR"]]
            .rename(columns={"MCC": "MCC_conf", "ERROR": "ERROR_conf"})
        )
        b = (
            dg[dg["risk_method"] == "learned_failure_risk"]
            [["seed", "fold", "MCC", "ERROR"]]
            .rename(columns={"MCC": "MCC_learned", "ERROR": "ERROR_learned"})
        )
        m = a.merge(b, on=["seed", "fold"])

        for metric in ["MCC", "ERROR"]:
            x = m[f"{metric}_learned"].to_numpy(float)
            y = m[f"{metric}_conf"].to_numpy(float)
            valid = np.isfinite(x) & np.isfinite(y)

            if valid.sum() >= 3 and np.any(np.abs(x[valid] - y[valid]) > 0):
                stat, pval = wilcoxon(
                    x[valid], y[valid], zero_method="wilcox"
                )
            else:
                stat, pval = np.nan, np.nan

            compare_rows.append({
                "dataset": dataset,
                "model": model,
                "coverage": float(coverage),
                "metric": metric,
                "n_runs": int(valid.sum()),
                "confidence_mean": float(np.nanmean(y)),
                "learned_risk_mean": float(np.nanmean(x)),
                "delta_learned_minus_confidence": float(np.nanmean(x - y)),
                "wilcoxon_stat": stat,
                "p_value": pval,
            })

    risk_compare = pd.DataFrame(compare_rows)
    risk_compare["p_holm"] = np.nan

    for _, idx in risk_compare.groupby(
        ["dataset", "coverage", "metric"]
    ).groups.items():
        risk_compare.loc[idx, "p_holm"] = holm_adjust(
            risk_compare.loc[idx, "p_value"].to_numpy(float)
        )

    risk_compare.to_csv(
        RISK_ROOT / "TABLE_confidence_vs_learned_risk.csv",
        index=False,
    )

    display(
        risk_compare[
            np.isclose(risk_compare["coverage"], 0.50)
        ].sort_values(["dataset", "metric", "model"]).round(4)
    )

In [ ]:
# ============================================================
# 15. Figure: learned-risk advantage at 50% coverage
# ============================================================

if RUN["risk_abstention"] and RUN["make_figures"]:
    src = risk_compare[
        (np.isclose(risk_compare["coverage"], 0.50))
        & (risk_compare["metric"] == "ERROR")
    ].copy()

    # For error, negative is good: learned-risk abstention leaves fewer errors.
    src["label"] = src["dataset"] + " / " + src["model"].str.replace(
        "ZHMolGraph-Best120", "ZHMolGraph", regex=False
    )
    src = src.sort_values("delta_learned_minus_confidence")

    fig, ax = plt.subplots(figsize=(3.45, 4.1))
    y = np.arange(len(src))
    ax.barh(y, src["delta_learned_minus_confidence"])
    ax.axvline(0, linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(src["label"])
    ax.set_xlabel(
        "Error difference at 50% coverage\n"
        "(learned risk − confidence)"
    )
    ax.set_title("Operational value of learned failure risk")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    save_figure_with_source(
        fig,
        src.drop(columns="label"),
        "FIG_LEARNED_RISK_ERROR_DELTA_50",
        caption=(
            "Difference in retained-set error at 50% coverage when predictions "
            "are ranked by learned protein-grouped failure risk rather than "
            "primary-model confidence. Negative values favor the learned "
            "failure-risk policy."
        ),
        interpretation=(
            "This figure directly tests whether the secondary reliability "
            "model produces an operational benefit beyond confidence-based "
            "abstention."
        ),
    )

# Cross-experiment summary


In [ ]:
# ============================================================
# Build compact cross-experiment summary
# ============================================================

summary_parts = []

if RUN["protein_balanced"]:
    s = (
        pb_runs.groupby(["dataset", "model"])
        .agg(
            edge_AUROC=("edge_AUROC", "mean"),
            protein_balanced_AUROC=("protein_balanced_AUROC", "mean"),
            edge_MCC=("edge_MCC", "mean"),
            protein_balanced_MCC=("protein_balanced_MCC", "mean"),
        )
        .reset_index()
    )
    s["PB_AUROC_delta"] = (
        s["protein_balanced_AUROC"] - s["edge_AUROC"]
    )
    s["PB_MCC_delta"] = (
        s["protein_balanced_MCC"] - s["edge_MCC"]
    )
    summary_parts.append(s)

if RUN["failure_ablation"]:
    a = failure_ablation[
        failure_ablation["feature_set"].isin(
            ["confidence", FINAL_FAILURE_FEATURE_SET]
        )
    ].pivot(
        index=["dataset", "model"],
        columns="feature_set",
        values="failure_AUROC",
    ).reset_index()

    a = a.rename(columns={
        "confidence": "failure_AUROC_confidence",
        FINAL_FAILURE_FEATURE_SET: "failure_AUROC_final",
    })
    a["failure_AUROC_gain"] = (
        a["failure_AUROC_final"] - a["failure_AUROC_confidence"]
    )

    if summary_parts:
        summary_parts[0] = summary_parts[0].merge(
            a, on=["dataset", "model"], how="outer"
        )
    else:
        summary_parts.append(a)

if RUN["secondary_leakage"]:
    l = leakage_results.pivot(
        index=["dataset", "model"],
        columns="split_strategy",
        values="failure_AUROC",
    ).reset_index().rename(columns={
        "record": "failure_AUROC_record_split",
        "pair": "failure_AUROC_pair_grouped",
        "protein": "failure_AUROC_protein_grouped",
    })
    l["record_minus_protein_AUROC"] = (
        l["failure_AUROC_record_split"]
        - l["failure_AUROC_protein_grouped"]
    )
    l["pair_minus_protein_AUROC"] = (
        l["failure_AUROC_pair_grouped"]
        - l["failure_AUROC_protein_grouped"]
    )

    if summary_parts:
        summary_parts[0] = summary_parts[0].merge(
            l, on=["dataset", "model"], how="outer"
        )
    else:
        summary_parts.append(l)

if RUN["lodo"]:
    ld = lodo_results[
        ["model", "target_dataset", "failure_AUROC", "failure_AUPRC"]
    ].rename(columns={
        "target_dataset": "dataset",
        "failure_AUROC": "LODO_failure_AUROC",
        "failure_AUPRC": "LODO_failure_AUPRC",
    })

    if summary_parts:
        summary_parts[0] = summary_parts[0].merge(
            ld, on=["dataset", "model"], how="outer"
        )
    else:
        summary_parts.append(ld)

if RUN["risk_abstention"]:
    rr = risk_compare[
        (np.isclose(risk_compare["coverage"], 0.50))
    ].pivot(
        index=["dataset", "model"],
        columns="metric",
        values="delta_learned_minus_confidence",
    ).reset_index().rename(columns={
        "MCC": "learned_minus_confidence_MCC_at50",
        "ERROR": "learned_minus_confidence_ERROR_at50",
    })

    if summary_parts:
        summary_parts[0] = summary_parts[0].merge(
            rr, on=["dataset", "model"], how="outer"
        )
    else:
        summary_parts.append(rr)

analysis_summary = summary_parts[0].sort_values(
    ["dataset", "model"]
).reset_index(drop=True)

analysis_summary.to_csv(
    OUTPUT_ROOT / "TABLE_posthoc_key_summary.csv",
    index=False,
)

display(analysis_summary.round(4))

# Reproducibility manifest and compact export


In [ ]:
# ============================================================
# 18. Save reproducibility metadata
# ============================================================

import sklearn
import scipy
import matplotlib

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "artifact_source": str(ARTIFACT_SOURCE),
    "prediction_rows": int(len(pred)),
    "primary_run_ids": int(pred["run_id"].nunique()),
    "datasets": sorted(pred["dataset"].unique().tolist()),
    "models": sorted(pred["model"].unique().tolist()),
    "protocols": sorted(pred["protocol"].unique().tolist()),
    "random_state": RANDOM_STATE,
    "n_meta_splits": N_META_SPLITS,
    "rf_params": RF_PARAMS,
    "failure_feature_sets": ABLATION_FEATURES,
    "final_failure_feature_set": FINAL_FAILURE_FEATURE_SET,
}

with open(OUTPUT_ROOT / "environment_and_analysis_config.json", "w") as f:
    json.dump(environment, f, indent=2)

manifest_rows = []
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        manifest_rows.append({
            "relative_path": str(p.relative_to(OUTPUT_ROOT)),
            "size_bytes": p.stat().st_size,
        })

pd.DataFrame(manifest_rows).to_csv(
    OUTPUT_ROOT / "MANIFEST.csv",
    index=False,
)

print("Saved reproducibility metadata.")

In [ ]:
# ============================================================
# 19. Create compact downloadable ZIP
# ============================================================

import shutil

zip_base = Path("/kaggle/working/RPI_Reliability_Posthoc_ARTIFACTS")
zip_file = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT,
)

print("Created:", zip_file)
print("Size (MB):", round(Path(zip_file).stat().st_size / 1024**2, 2))
print("\nDownload this ZIP after the notebook finishes.")